In [27]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [28]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [29]:
# run_pattern = "mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "mswep_precipitation_chirps_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
run_pattern = "mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/validation/model_epoch030/validation_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_1704_041621', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_1704_054334', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_1704_071030', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_1704_083712', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_1704_100403', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_1704_113108', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_1704_125817', 'mswep_precipitation_chirps_v3_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_1704_142519']


In [30]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01411300': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.75 1.99 2.85 ... 0.46 0.55
       streamflow_sim  (date, time_step) float32 15kB 1.658 2.027 ... 0.6289 0.7137}},
 'camels_01487000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 1989-10-01 ... 1999-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.43 1.88 2.14 ... 1.47 1.58
       streamflow_sim  (date, time_step) float32 15kB 2.12 2.696 ... 1.183 1.491}},
 'camels_01491000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinate

In [31]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    all_metrics[basin_id] = calculate_metrics(
        obs=xr_ds['streamflow_obs'], #['QObs_mm_d_obs'],
        sim=xr_ds['streamflow_sim'], #['QObs_mm_d_sim'],
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'

df_metrics

,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
camels_01411300,0.817283,0.311190,0.557844,0.739339,0.793807,0.924216,0.859696,-0.141020,-24.224728,7.642509e+00,2.005500e+01,0.050000,0.320755,27.049074
camels_01487000,0.835234,0.226205,0.475610,0.858239,1.122103,0.933586,1.027862,0.028459,10.923643,1.227770e+01,-6.903625e+01,0.166667,0.309524,22.669277
camels_01491000,0.652701,1.527043,1.235736,0.627718,0.682490,0.820765,1.075210,0.042828,-34.645157,8.725691e-01,5.566381e+01,0.277778,0.372549,55.161179
camels_01644000,0.748586,0.965665,0.982682,0.848339,0.932245,0.867764,0.969602,-0.016470,1.167373,-3.141030e+00,6.692915e+01,0.227273,0.245283,34.699406
camels_01664000,0.588958,1.855745,1.362257,0.751604,0.933180,0.784159,0.896808,-0.061148,-5.678035,-6.277072e+00,6.457755e+01,0.142857,0.372881,39.470467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
camels_08200000,0.796339,1.204704,1.097590,0.618900,0.762154,0.905492,0.717626,-0.055052,-34.822006,-4.070788e+01,8.743784e+01,0.250000,0.324324,52.246105
camels_08202700,0.713173,0.456149,0.675388,-3.055599,1.190325,0.904495,5.050004,0.152414,105.115982,1.074880e+08,-1.621026e+11,0.333333,0.214286,80.414124
camels_09484600,0.090699,0.003029,0.055036,0.127671,0.622331,0.390937,0.502640,-0.092687,-43.088737,9.180191e+08,-0.000000e+00,0.307692,0.534884,71.681694


In [32]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(f"./ensemble_metrics/{save_name}.csv")

In [33]:
df_metrics.median()

NSE              0.736964
MSE              1.621643
RMSE             1.273418
KGE              0.733772
Alpha-NSE        0.858741
Pearson-r        0.871129
Beta-KGE         0.957602
Beta-NSE        -0.019092
FHV            -13.703603
FMS            -11.306626
FLV              9.342597
Peak-Timing      0.292857
Missed-Peaks     0.350000
Peak-MAPE       42.540667
dtype: float64